In [64]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)
import tensorflow as tf
from sklearn import preprocessing
import numpy as np
import random
import matplotlib.pyplot as plt
import sys
import pandas as pd
from AES import*
from Model.CNN import cnn_classifier
from sklearn.model_selection import train_test_split
from collections import Counter
from scipy.stats import mode
from utils.LoadData import load_CW_Source,load_CW_Target
from sklearn.utils import shuffle

**Step 1-2: Establish the Mapping for All Bytes and Identify the Key-Related S-box**

In [65]:
def getBox(num,best_byte):
    # Initialize mapping matrices
    pk_v_box = np.zeros((256, 256))  # Stores p^k to v mapping
    p_v_box = np.zeros((16, 256))    # Stores p to v mapping
    
    # Process each byte position (0-15)
    for byte in range(16):
        # Load profiling traces (training data)
        profiling_traces, _, _, _, _, _ = load_CW_Source(
            in_file=profiling_Data_path,
            sec=45000,  # Fixed security parameter
            byte=byte
        )
        # Load attack traces and associated data
        X_attack, label_V, p_attack = load_CW_Target(
            in_file=Target_Data_path,
            byte=byte
        )
        
        # Load and configure model
        model = cnn_classifier(input_size=600)
        model.reset_states()
        model_name = f'Source_Model_byte{byte}_D1.h5'
        model.load_weights(model_path + model_name)

        # Shuffle datasets while maintaining correspondence
        X_attack, label_V, p_attack = shuffle(X_attack, label_V, p_attack)

        # Slice datasets to specified size
        X_attack_shuffle = X_attack[:num]
        label_V_shuffle = label_V[:num]
        p_attack_shuffle = p_attack[:num]

        # Data preprocessing pipeline
        # 1. Standardization (zero-mean, unit-variance)
        scaler = preprocessing.StandardScaler()
        profiling_traces = scaler.fit_transform(profiling_traces)
        X_attack_shuffle = scaler.transform(X_attack_shuffle)
        
        # 2. Normalization to [0,1] range
        scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
        profiling_traces = scaler.fit_transform(profiling_traces)
        X_attack_shuffle = scaler.transform(X_attack_shuffle)

        # Generate model predictions
        predictions = model.predict(X_attack_shuffle)

        # Process prediction results
        A = np.squeeze(predictions)  # Remove singleton dimensions
        B = np.squeeze(p_attack_shuffle)
        df = pd.DataFrame({'pred': list(A), 'plaintext': list(B)})

        # Aggregate predictions by plaintext value
        sum_by_plaintext = df.groupby('plaintext')['pred'].sum()

        # Build pk_v_box mapping (byte 0 only)
        if byte == best_byte:
            for k in range(256):  # All possible key values
                for i in range(256):  # All possible input values
                    if i in sum_by_plaintext:
                        # Xor-based mapping: i^k -> argmax
                        pk_v_box[k, i ^ k] = np.argmax(sum_by_plaintext[i])
                    else:
                        pk_v_box[k, i ^ k] = -1  # Invalid entry

        # Build p_v_box mapping for current byte
        for j in range(256):  # All possible plaintext values
            if j in sum_by_plaintext:
                p_v_box[byte, j] = np.argmax(sum_by_plaintext[j])
            else:
                p_v_box[byte, j] = -1  # Mark missing entries

    return pk_v_box, p_v_box

<b> Evaluation of the Number of Recovered S-boxes

In [66]:
def attackForSbox(num,SboxType,best_byte):
    # Predefined correct key (16 bytes)
    key = [0x3F,0x1C,0x77,0xC5,0xA8,0x6E,0x5A,0xF1,0x19,0xA4,0x07,0x3F,0x51,0xFD,0xAE,0xA7]
    
    # Get key-dependent S-box mappings
    pk_v_box, p_v_box = getBox(num,best_byte)
    # ----- S-box Validation Test -----
    tnum = 0
    # Verify consistency between Skinny_Sbox and generated pk_v_box
    if(SboxType=='SM4'):
        Sbox=SM4_Sbox
    elif(SboxType=='Skinny'):
        Sbox=Skinny_Sbox
    else:
        Sbox=AES_Sbox

    for i in range(256):
        if (Sbox[i] - pk_v_box[key[best_byte], i]) != 0:
            # print(i)
            tnum += 1  # Count mismatched entries
    print('wrong num',tnum)
    
    return 256-tnum

def evaluate_Recovered_Sboxes(attack_range,SboxType,best_byte):
    """Evaluate the accuracy of attackForSbox function
    Args:
        attack_range: Range of attack numbers to test (e.g., range(2000, 10001, 500))
    
    Returns:
        List of tuples containing (attack_num, accuracy_rate)
    """
    num_trials = 10  # Number of trials per attack number
    
    SR_results = []
    
    # Test each attack number in the range
    for attack_num in attack_range:
        success_count = 0
        
        # Run multiple trials for statistical significance
        for _ in range(num_trials):
            result = attackForSbox(attack_num,SboxType,best_byte)
            success_count += result
        
        # Calculate success rate
        accuracy = success_count / num_trials
        SR_results.append((attack_num, accuracy))
        
        print(f"Attack Num: {attack_num}, Sbox_num: {accuracy:.2f}")
    
    return SR_results

<b>Source Model-Target(SM4-sbox)

In [67]:
# Main parameter initialization
best_byte=4  #the byte with the highest prediction accuracy is selected for S-box recovery
profiling_Data_path='./Dataset/mask_AES/AES_Sbox/'
Target_Data_path='./Dataset/mask_AES/SM4_Sbox/'
model_path = './Model/mask_AES/'

In [ ]:
SM4_SR=evaluate_Recovered_Sboxes(range(2000, 12001,1000),'SM4',best_byte)

<b><b>Source Model-Target(Skinny-sbox)

In [ ]:
# Main parameter initialization
best_byte=4
profiling_Data_path='./Dataset/mask_AES/AES_Sbox/'
Target_Data_path='./Dataset/mask_AES/Skinny_Sbox/'
model_path = './Model/mask_AES/'

In [ ]:
Skinny_SR=evaluate_Recovered_Sboxes(range(5000, 12001,1000),'Skinny',best_byte=4)

In [ ]:
x = range(5000, 12001,1000)
plt.plot(x, [y for _, y in SM4_SR],color='b', marker='^',label='Source Model-Target(SM4 SBOX)')
plt.plot(x, [y for _, y in Skinny_SR],color='gold',marker='x',label='Source Model-Target(SKINNY SBOX)')
plt.grid(True)  

plt.xlabel('Number of the traces', fontsize=16)
plt.ylabel("Recovered S-box Element Count", fontsize=16)
plt.legend(fontsize=12)
plt.tick_params(labelsize=14)
plt.tight_layout()
# plt.savefig(d_out+'AES-X_attackACC.pdf')
plt.savefig('AES-Sbox_num.pdf')
plt.show()